# Inference and Autoregressive Generation

> During training, the model sees the correct preceding context (Teacher Forcing), so all positions can be computed in parallel. During inference, there is no correct answer -- the model must first generate one token, append it to the input, then generate the next.
>
> This section implements the core strategies of autoregressive generation: greedy decoding, temperature sampling, top-k/top-p truncation, and beam search. We first train a small model so that the effect of each strategy is directly visible in the output.

Autoregressive Generation is the fundamental way all current LLMs produce text. The model generates the first token, appends it to the input, generates the second, appends it back, and so on in a loop.

Only one token is generated at a time, but the entire accumulated sequence must go through Attention computation again each step. This serial nature is the root cause of slow LLM inference, and it is also the key reason the model can dynamically adjust its output based on what has already been generated.

## 1. The Fundamental Difference Between Inference and Training

```
Training:  Has answer -> All positions compute loss in parallel -> Teacher Forcing
Inference: No answer  -> Must generate tokens one by one      -> Autoregressive
```

Autoregressive means: use the model's own output as the input for the next step.

```
Step 1: Input [BOS]           -> Model predicts -> I
Step 2: Input [BOS, I]        -> Model predicts -> love
Step 3: Input [BOS, I, love]  -> Model predicts -> you
Step 4: Input [BOS, I, love, you] -> Model predicts -> EOS -> Stop
```

Like a snake eating its own tail -- it keeps getting longer.

## 2. Training a Model That Shows Visible Effects

To make the effects of temperature, top-k, and other strategies clearly visible, we need a model whose output is not completely deterministic (there is probability spread) but also not completely random. We use training data with patterns that are regular but not unique: given a few tokens, there are multiple reasonable continuations.

Here we use a "multi-path pattern": each position has 2-3 reasonable successors, and the model assigns different probabilities to different successors. This way, the effect of temperature can be directly observed in the output.

In [ ]:
import torch
import torch.nn as nn

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=2, num_layers=2):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(64, d_model)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=num_heads,
                dim_feedforward=4*d_model, batch_first=True, activation='relu')
            for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        batch, seq = x.shape
        pos = torch.arange(seq, device=x.device).unsqueeze(0).expand(batch, -1)
        h = self.token_emb(x) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(seq, device=x.device)
        for block in self.blocks:
            h = block(h, src_mask=mask, is_causal=True)
        return self.lm_head(h)

VOCAB = 16

def make_multi_path_data(n=800, seq_len=10):
    # Multiple paths: starting from token 1, there are two possible paths
    # Path A: 1->2->3->4->5->6->7->8->1...  (70%)
    # Path B: 1->2->9->10->11->12->7->8->1... (30%)
    # Branch point at position 2: token 3 or 9
    data = []
    for i in range(n):
        path = i % 10  # about 70% take path A
        seq = [1, 2]
        if path < 7:
            seq.extend([3, 4, 5, 6])
        else:
            seq.extend([9, 10, 11, 12])
        seq.extend([7, 8])
        data.append(seq[:seq_len])
    return torch.tensor(data)

train_data = make_multi_path_data()
print(f'Training data: {train_data.shape}')
print(f'Path A sample: {train_data[0].tolist()}')
print(f'Path B sample: {train_data[7].tolist()}')
print(f'Branch point: position 2 -> token 3 (70%) or token 9 (30%)')

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = SimpleGPT(vocab_size=VOCAB, d_model=32, num_heads=2, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

BATCH_SIZE = 16
NUM_EPOCHS = 30

print(f'Training {NUM_EPOCHS} epochs...')
model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for i in range(0, len(train_data), BATCH_SIZE):
        batch = train_data[i:i+BATCH_SIZE]
        input_ids = batch[:, :-1]
        targets = batch[:, 1:]
        logits = model(input_ids)
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, VOCAB), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {total_loss:.4f}')

print('Training complete!')

In [ ]:
# Check the model's probability distribution at the branch point
import torch
import torch.nn.functional as F

model.eval()
with torch.no_grad():
    # Input [1, 2], check prediction at position 2
    logits = model(torch.tensor([[1, 2]]))
    probs = F.softmax(logits[0, -1, :], dim=-1)

print('After input [1, 2], model probability distribution for next token:')
for tok_id in range(VOCAB):
    p = probs[tok_id].item()
    if p > 0.01:
        bar = '#' * int(p * 50)
        print(f'  token {tok_id:2d}: {p:.3f} {bar}')

print(f'\ntoken 3 (Path A): {probs[3].item():.1%}')
print(f'token 9 (Path B): {probs[9].item():.1%}')
print(f'\nThe model learned the probability distribution of both paths! Temperature adjustment will change which path to take.')

## 3. Greedy Decoding

At each step, select the token with the highest probability. The advantage is determinism and speed -- the same input always produces the same output. The disadvantage is that once a token is chosen, the opportunity to explore other branches is permanently lost.

In [ ]:
import torch

def generate_greedy(model, input_ids, max_new_tokens=20, eos_id=None):
    model.eval()
    generated = input_ids.clone()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(generated)
            next_logits = logits[0, -1, :]
            next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
            if eos_id is not None and next_token.item() == eos_id:
                break
    return generated

prompt = torch.tensor([[1, 2]])
result = generate_greedy(model, prompt, max_new_tokens=12)
print(f'Greedy result: {result[0].tolist()}')
print(f'\nGreedy always picks the highest probability, so it always takes Path A (token 3).')
print(f'Even though Path B has 30% probability, Greedy will never choose it.')

## 4. Temperature: Controlling Randomness

Temperature controls the shape of the probability distribution by scaling logits:

```
probability = softmax(logits / temperature)

temperature = 0.1 -> very peaked distribution -> almost greedy
temperature = 1.0 -> original distribution
temperature = 2.0 -> flatter distribution  -> more random
```

In [ ]:
# Effect of temperature on the branch point
import torch
import torch.nn.functional as F

model.eval()
with torch.no_grad():
    logits = model(torch.tensor([[1, 2]]))[0, -1, :]

print('=== Effect of Temperature on Branch Point Probabilities ===')
print(f'{"Temperature":>12}  {"P(token 3)":>10}  {"P(token 9)":>10}  {"Ratio 3/9":>8}')
print('-' * 48)
for T in [0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 5.0]:
    probs = F.softmax(logits / T, dim=-1)
    p3 = probs[3].item()
    p9 = probs[9].item()
    ratio = p3 / p9 if p9 > 0 else float('inf')
    print(f'{T:>12.1f}  {p3:>10.3f}  {p9:>10.3f}  {ratio:>8.1f}')

print(f'\nLow temperature: token 3 is almost 100%, equivalent to greedy')
print(f'High temperature: the gap between token 3 and 9 shrinks, model more likely to explore Path B')

In [ ]:
# Visualization
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
example_logits = logits.clone()
for ax, T in zip(axes, [0.2, 0.7, 1.5, 5.0]):
    probs = F.softmax(example_logits / T, dim=-1)
    top_k_probs, top_k_idx = torch.topk(probs, 6)
    ax.bar(range(6), top_k_probs.numpy())
    ax.set_xticks(range(6))
    ax.set_xticklabels([str(i.item()) for i in top_k_idx])
    ax.set_title(f'T={T}')
    ax.set_ylim(0, 1)
plt.suptitle('Effect of Temperature on Branch Point Probability Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Actual generation results with different temperatures
import torch
import torch.nn.functional as F

print('Same prompt [1, 2], generation results with different temperatures:')
print()

for T in [0.1, 0.5, 1.0, 1.5, 2.0]:
    results = []
    for seed in range(5):
        torch.manual_seed(seed)
        generated = torch.tensor([[1, 2]])
        model.eval()
        with torch.no_grad():
            for _ in range(12):
                logits = model(generated)[0, -1, :] / T
                probs = F.softmax(logits, dim=-1)
                token = torch.multinomial(probs, 1).unsqueeze(0)
                generated = torch.cat([generated, token], dim=1)
        results.append(generated[0, 2:6].tolist())  # Check branch point choice
    paths = ['A' if r[0] == 3 else 'B' for r in results]
    print(f'T={T:.1f}: {results}  Path: {paths}')

print(f'\nLow temperature: almost all take Path A')
print(f'High temperature: Path B starts appearing, diversity increases')

## 5. Top-k and Top-p Sampling

Even with temperature turned up, there are still many extremely low-probability tokens. If one of them happens to be sampled, generation quality will suffer.

**Top-k**: only sample from the k highest-probability tokens. **Top-p** (nucleus sampling): accumulate probabilities from highest to lowest, stop when reaching p. Top-p is adaptive -- when the distribution is concentrated it selects few tokens, when dispersed it selects more.

In [ ]:
import torch
import torch.nn.functional as F

def generate_sampled(model, input_ids, max_new=20, temperature=1.0, top_k=None, top_p=None, seed=42):
    torch.manual_seed(seed)
    model.eval()
    generated = input_ids.clone()
    with torch.no_grad():
        for _ in range(max_new):
            logits = model(generated)[0, -1, :]
            logits = logits / max(temperature, 0.01)

            if top_k is not None:
                topk_vals, _ = torch.topk(logits, top_k)
                logits[logits < topk_vals[-1]] = float('-inf')

            if top_p is not None:
                sorted_l, sorted_idx = torch.sort(logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_l, dim=-1), dim=-1)
                remove = cum_probs > top_p
                remove[0] = False
                logits[sorted_idx[remove]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            token = torch.multinomial(probs, 1).unsqueeze(0)
            generated = torch.cat([generated, token], dim=1)
    return generated

print('=== Top-k vs Top-p Comparison ===')
prompt = torch.tensor([[1, 2]])
print(f'\nSame prompt [1, 2], temperature=1.0, seed=42:')
for k in [1, 2, 4, None]:
    r = generate_sampled(model, prompt, max_new=12, temperature=1.0, top_k=k, seed=42)
    label = f'top_k={k}' if k else 'no filter'
    print(f'  {label:12s}: {r[0].tolist()}')

for p in [0.5, 0.9, 0.99, None]:
    r = generate_sampled(model, prompt, max_new=12, temperature=1.0, top_p=p, seed=42)
    label = f'top_p={p}' if p else 'no filter'
    print(f'  {label:12s}: {r[0].tolist()}')

## 6. Beam Search

Greedy picks the best at each step, but local optimum does not equal global optimum. Beam Search maintains K paths simultaneously and selects the K highest-scoring paths from all candidates at each step.

Suitable for: translation, summarization, and other tasks with "clear answers". Not suitable for: creative writing -- beam search makes the output boring.

In [ ]:
import torch
import torch.nn.functional as F

def beam_search(model, input_ids, beam_size=3, max_new=12):
    beams = [(0.0, input_ids.clone())]
    for _ in range(max_new):
        candidates = []
        for score, seq in beams:
            with torch.no_grad():
                logits = model(seq)[0, -1, :]
            log_probs = F.log_softmax(logits, dim=-1)
            top_probs, top_idx = torch.topk(log_probs, beam_size)
            for i in range(beam_size):
                new_seq = torch.cat([seq, top_idx[i].unsqueeze(0).unsqueeze(0)], dim=1)
                candidates.append((score + top_probs[i].item(), new_seq))
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_size]
    best_score, best_seq = beams[0]
    return best_seq, best_score

prompt = torch.tensor([[1, 2]])
print('=== Beam Search ===')
for bs in [1, 2, 3, 5]:
    result, score = beam_search(model, prompt, beam_size=bs, max_new=10)
    label = 'Greedy' if bs == 1 else f'Beam k={bs}'
    print(f'{label:12s}: {result[0].tolist()}  score={score:.2f}')

## 7. Repetition Penalty

LLMs are prone to falling into repetitive loops during generation. Repetition Penalty reduces the logit of tokens that have already appeared, breaking the loop.

Penalty formula: if logit > 0, divide by penalty; otherwise multiply by penalty. penalty > 1 means applying the penalty.

In [ ]:
import torch
import torch.nn.functional as F

def apply_repetition_penalty(logits, token_ids, penalty=1.2):
    if penalty == 1.0:
        return logits
    for tid in set(token_ids.tolist()):
        score = logits[tid]
        logits[tid] = score / penalty if score > 0 else score * penalty
    return logits

# Use a prompt that tends to repeat
# Run multiple times and measure repetition rate
print('=== Repetition Penalty Effect ===')
print(f'{"penalty":>8}  {"Generated Sequence":>40}  {"Unique Ratio":>6}')
print('-' * 60)

prompt = torch.tensor([[1, 2, 3]])
for penalty in [1.0, 1.1, 1.3, 1.5, 2.0]:
    torch.manual_seed(42)
    generated = prompt.clone()
    model.eval()
    with torch.no_grad():
        for _ in range(15):
            logits = model(generated)[0, -1, :].clone()
            logits = apply_repetition_penalty(logits, generated[0], penalty)
            probs = F.softmax(logits / 0.8, dim=-1)
            token = torch.multinomial(probs, 1).unsqueeze(0)
            generated = torch.cat([generated, token], dim=1)
    tokens = generated[0].tolist()
    unique_ratio = len(set(tokens)) / len(tokens)
    print(f'{penalty:>8.1f}  {str(tokens):>40}  {unique_ratio:>5.1%}')

## 8. Chat Templates and System Prompt

The generation examples above all follow the pattern of "given some text, continue writing." In practice, LLMs are used in a conversational format. Chat templates stitch multi-turn conversations into a single piece of text that the model can understand.

Different models use different templates. Using the wrong template leads to degraded response quality. This is why HuggingFace's `tokenizer.apply_chat_template()` is useful -- it handles the formatting automatically.

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': 'What is the attention mechanism?'},
    {'role': 'assistant', 'content': 'The attention mechanism is a...'},
    {'role': 'user', 'content': 'Can you give an example?'},
]

def apply_chatml(msgs):
    parts = []
    for m in msgs:
        parts.append(f'<|im_start|>{m["role"]}\n{m["content"]}<|im_end|>')
    parts.append('<|im_start|>assistant')
    return '\n'.join(parts)

def apply_llama(msgs):
    parts = []
    for m in msgs:
        if m['role'] == 'system':
            parts.append(f'<<SYS>>\n{m["content"]}<</SYS>>\n\n')
        elif m['role'] == 'user':
            parts.append(f'[INST] {m["content"]} [/INST]')
        else:
            parts.append(f' {m["content"]} </s><s>')
    return ''.join(parts)

def apply_alpaca(msgs):
    system = next((m['content'] for m in msgs if m['role'] == 'system'), '')
    text = 'Below is an instruction. Write a response.\n\n'
    if system: text += f'### System\n{system}\n\n'
    for m in msgs:
        if m['role'] == 'user': text += f'### Instruction\n{m["content"]}\n\n'
        elif m['role'] == 'assistant': text += f'### Response\n{m["content"]}\n\n'
    text += '### Response\n'
    return text

print('=== ChatML (Qwen/Yi) ===')
print(apply_chatml(messages))
print('\n=== Llama Chat ===')
print(apply_llama(messages))
print('\n=== Alpaca ===')
print(apply_alpaca(messages)[:200] + '...')

### Tips for Writing System Prompts

A few principles:

1. **Define the role clearly**: tell the model "who you are"
2. **Set boundaries**: "what not to do" is more effective than "what to do"
3. **Specify format**: "output format is JSON" or "use markdown tables"
4. **Give examples**: one example is worth a hundred rules (few-shot)

Bad System Prompt: `You are an assistant, please help the user.`

Good System Prompt:
```
You are a Python debugging expert.
Users will give you error messages and related code.
Your task:
1. Find the root cause (summarize in one sentence)
2. Provide the fix code
3. Explain why the fix works
Reply in English. Wrap code in markdown code blocks.
```

## 9. The Complete Generation Pipeline

Connecting all concepts from Part 1 to now in one flow:

```
User inputs text
    | Tokenizer.encode()          <- Part 1 & 2
token ID sequence
    | Embedding + Position         <- Part 3
vector sequence
    | N x Transformer Block        <- Part 4
logits
    | Sampling strategy            <- This section
generate tokens one by one
    | Tokenizer.decode()           <- Part 1 & 2
output text
```

GPT-4, Claude, and Gemini are all super-scaled versions of this pipeline.

In [ ]:
# Complete generation function: integrating all strategies
import torch
import torch.nn.functional as F

def generate(model, input_ids, max_new=30, temperature=1.0, top_k=None, top_p=None,
            repetition_penalty=1.0, eos_id=None, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    generated = input_ids.clone()
    with torch.no_grad():
        for _ in range(max_new):
            logits = model(generated)[0, -1, :].clone()

            # 1. Repetition penalty
            if repetition_penalty != 1.0:
                logits = apply_repetition_penalty(logits, generated[0], repetition_penalty)

            # 2. Temperature
            logits = logits / max(temperature, 0.01)

            # 3. Top-k
            if top_k is not None:
                topk_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < topk_vals[-1]] = float('-inf')

            # 4. Top-p
            if top_p is not None:
                sorted_l, sorted_idx = torch.sort(logits, descending=True)
                cum = torch.cumsum(F.softmax(sorted_l, dim=-1), dim=-1)
                remove = cum > top_p
                remove[0] = False
                logits[sorted_idx[remove]] = float('-inf')

            # 5. Sample
            probs = F.softmax(logits, dim=-1)
            token = torch.multinomial(probs, 1).unsqueeze(0)
            generated = torch.cat([generated, token], dim=1)

            # 6. EOS check
            if eos_id is not None and token.item() == eos_id:
                break
    return generated

print('Complete generation function defined!')
print('\nIndustry-standard execution order:')
print('  1. Repetition Penalty -> penalize already-seen tokens')
print('  2. Temperature -> adjust distribution shape')
print('  3. Top-k / Top-p -> truncate low probabilities')
print('  4. Sampling -> sample from final distribution')
print('  5. EOS check -> whether to stop')

In [ ]:
# Final comparison: generation results with different configurations
import torch

prompt = torch.tensor([[1, 2]])
configs = [
    ('Greedy',        dict(temperature=0.01, top_k=1)),
    ('T=0.5',         dict(temperature=0.5)),
    ('T=1.0',         dict(temperature=1.0)),
    ('T=1.5',         dict(temperature=1.5)),
    ('T=1.0, k=2',   dict(temperature=1.0, top_k=2)),
    ('T=1.0, p=0.9', dict(temperature=1.0, top_p=0.9)),
    ('T=0.8, rep=1.3', dict(temperature=0.8, repetition_penalty=1.3)),
]

print(f'{"Config":>16}  Generation Result')
print('-' * 65)
for name, cfg in configs:
    r = generate(model, prompt, max_new=12, seed=42, **cfg)
    tokens = r[0].tolist()
    path = 'A' if 3 in tokens[2:5] else ('B' if 9 in tokens[2:5] else '?')
    print(f'{name:>16}  {tokens}  Path {path}')

## Summary

- Difference between inference and training: training has answers (parallel), inference has no answers (serial)
- Greedy picks the highest probability -- deterministic but uninteresting
- Temperature controls randomness -- low temperature is deterministic, high temperature is diverse
- Top-k limits the number of candidates, Top-p truncates by cumulative probability
- Beam Search maintains multiple paths -- suitable for translation/summarization
- Repetition Penalty breaks repetitive loops
- Chat templates stitch multi-turn conversations into model-readable text
- System Prompt guides the model through role and format requirements
- Industry-standard sampling execution order: penalty -> temperature -> top-k/p -> sample -> eos

Next section: break down the causes of slow inference, learn acceleration techniques like KV Cache, quantization, and FlashAttention.

## Exercises

**Exercise 1: Temperature Calculation**

Given logits = [2.0, 1.0, 0.5] and temperature = 0.5, what are the scaled logits?

Hint: logits / temperature

In [ ]:
import torch
import torch.nn.functional as F

logits = torch.tensor([2.0, 1.0, 0.5])
T = 0.5
scaled = logits / T
probs = F.softmax(scaled, dim=-1)
print(f'Scaled: {scaled.tolist()}')
print(f'Probabilities: {probs.tolist()}')
assert torch.allclose(scaled, torch.tensor([4.0, 2.0, 1.0]))
print('Exercise 1 passed!')

**Exercise 2: Top-k Filtering**

Given logits = [0.1, 2.0, 0.5, 3.0, 0.01] and top_k=2, which positions are kept?

Hint: find the positions of the 2 largest values.

In [ ]:
import torch

logits = torch.tensor([0.1, 2.0, 0.5, 3.0, 0.01])
topk_vals, topk_idx = torch.topk(logits, 2)
print(f'Kept positions: {topk_idx.tolist()} (values: {topk_vals.tolist()})')
assert topk_idx.tolist() == [3, 1]
print('Exercise 2 passed: position 3 (value 3.0) and position 1 (value 2.0)')

## References

- Holtzman et al., [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751), 2020 -- Nucleus Sampling (top-p)
- Fan et al., [Hierarchical Neural Story Generation](https://arxiv.org/abs/1805.04833), 2018 -- Top-k sampling
- Keskar et al., [CTRL: A Conditional Transformer Language Model](https://arxiv.org/abs/1909.05858), 2019 -- Temperature and repetition penalty
- Harvard NLP, [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)